In [ ]:
"""
to be covered in this project:
- What RAG is and the problems it solves: knowledge cutoffs, hallucinations,
  and domain-specific grounding

- How a RAG pipeline is structured: the four stages
  (query, retrieve, augment, generate) and how data flows between them

- Building a working pipeline: connecting ChromaDB and the OpenAI
  Chat Completions API into a functional system

- Prompt design for RAG: structuring prompts so the model uses provided
  documentation and handles unanswerable questions gracefully

- Source attribution: getting the model to cite its sources
"""

In [ ]:

import os
import json

import tiktoken

import cohere
import chromadb
# from openai import OpenAI
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
co = cohere.Client(api_key=os.getenv("COHERE_API_KEY"))
# oai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
client = chromadb.PersistentClient(path="Introduction_to_RAG/chroma_scoped")
collection = client.get_collection(name="git_docs_scoped")

print(f"ChromaDB collection: {collection.name}")
print(f"Document count: {collection.count()}")
print("Setup verified.")

/Users/sbamwite/Desktop/rags/env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


ChromaDB collection: git_docs_scoped
Document count: 334
Setup verified.


In [5]:

SYSTEM_PROMPT = """You are GitQuest, a Git support agent that helps \
developers use Git correctly and confidently.

Answer the user's question using ONLY the documentation provided below. \
Do not use knowledge from your training data.

Guidelines:
- Provide the exact command syntax as shown in the documentation
- Briefly explain what the command does and why it works
- If there are important options or variations shown in the docs, mention them
- If the provided documentation does not contain enough information to \
answer the question, say so explicitly rather than guessing or drawing \
on outside knowledge

End your answer with a SOURCES section listing only the chunk_ids you \
drew from, in this exact format:

SOURCES:
- chunk_id: <id> | <title>

Documentation:
{context}"""


In [ ]:
corpus = {}
with open("data/git_kb_corpus_scoped/corpus.jsonl", "r") as f:
    for line in f:
        chunk = json.loads(line)
        corpus[chunk["chunk_id"]] = chunk

MAX_REFORMULATIONS = 3

EXPANSION_PROMPT = """You are helping improve search over Git documentation.

Given a user's question, generate 2 or 3 alternative phrasings that use different vocabulary but ask the same thing. Use terminology that might appear in official Git documentation (command names, flags, technical terms).

Return ONLY the alternative phrasings, one per line, with no numbering, no bullet points, no explanation, and no blank lines.

User question: {query}"""


def retrieve(query, n_results=5):
    response = co.embed(
        texts=[query],
        model="embed-v4.0",
        input_type="search_query",
        embedding_types=["float"]
    )
    query_embedding = response.embeddings.float[0]
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )
    chunks = []
    for chunk_id, metadata, distance in zip(
        results["ids"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        chunk = corpus[chunk_id]
        chunks.append({
            "chunk_id": chunk_id,
            "text": chunk["text"],
            "title": chunk["title"],
            "source_type": metadata["source_type"],
            "command": metadata["command"],
            "distance": distance
        })
    return chunks

def expand_query(query):
    response = oai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": EXPANSION_PROMPT.format(query=query)}],
        # use temperature=0.3 to keep the reformulations focused and consistent rather than unpredictably creative.
        temperature=0.3,
    )

    raw = (response.choices[0].message.content or "").strip()
    lines = [line.strip() for line in raw.splitlines() if line.strip()]

    reformulations = []
    for line in lines:
        # Strip common bullet prefixes in case the model ignores "no bullets"
        for prefix in ("- ", "* ", "  "):
            if line.startswith(prefix):
                line = line[len(prefix):].strip()
                break

        # Strip simple numeric prefixes in case the model uses numbering
        if len(line) >= 3 and line[0].isdigit() and line[1] in (".", ")") and line[2] == " ":
            line = line[3:].strip()

        lowered = line.lower()

        # Skip commentary lines if the model adds preamble
        if lowered.startswith(("here are", "alternative", "reformulation", "sure", "note:", "explanation:")):
            continue

        # Skip exact echoes of the original query
        if lowered == query.lower():
            continue

        if line:
            reformulations.append(line)
        if len(reformulations) >= MAX_REFORMULATIONS:
            break

    return reformulations if reformulations else [query]


def expand_and_retrieve(query, n_results=10):
    reformulations = expand_query(query)
    all_queries = [query] + reformulations
    seen = {}
    for q in all_queries:
        for chunk in retrieve(q, n_results=n_results):
            cid = chunk["chunk_id"]
            if cid not in seen or chunk["distance"] < seen[cid]["distance"]:
                seen[cid] = chunk
    return sorted(seen.values(), key=lambda x: x["distance"])

def rerank(query, chunks, top_n=5):
    """
    
    - model="rerank-v3.5" is one of Cohere's reranking models.
      Unlike embedding models, reranking models don't produce vectors.
      They produce relevance scores, and rerank-v3.5 is optimized specifically for that task.
      NOTE: what is the cohere rerank API's relevance score depend on.
    
    - documents=documents passes the full text of each candidate chunk to the API.
      This is what enables the joint query-document scoring we discussed above.
      The model needs to read both to produce a meaningful relevance score.
    
    - result.index maps each result back to the original chunk in our list,
      so we can carry forward all the metadata we attached during retrieval.
    top_n=5 tells the reranker how many results to return. We're asking it to give us the 5 most relevant chunks from whatever candidate pool we pass in.
    """
    documents = [c["text"] for c in chunks]
    response = co.rerank(
        model="rerank-v3.5",
        query=query,
        documents=documents,
        top_n=top_n
    )
    reranked = []
    for result in response.results:
        chunk = chunks[result.index]
        reranked.append({
            **chunk,
            "rerank_score": result.relevance_score
        })
    return reranked

def build_context(chunks):
    context_parts = []
    for i, chunk in enumerate(chunks, start=1):
        context_parts.append(
            # f"[SOURCE {i}]\n"
            """
                remove label to reduce competing anchors and make chunk_id
                the single obvious citation handle.

                when both the chunk_id and a numbered label are present, a model
                may reach for whichever looks more natural in context.

                Removing the labels doesn't eliminate all citation inconsistency,
                but it removes one common cause of it.
            """
            f"chunk_id: {chunk['chunk_id']}\n"
            f"title: {chunk['title']}\n"
            f"source_type: {chunk['source_type']}\n"
            f"command: {chunk['command']}\n\n"
            f"{chunk['text']}"
        )
    return "\n\n---\n\n".join(context_parts)


SYSTEM_PROMPT = """You are GitQuest, a Git support agent that helps \
developers use Git correctly and confidently.

Answer the user's question using ONLY the documentation provided below. \
Do not use knowledge from your training data.

Guidelines:
- Provide the exact command syntax as shown in the documentation
- Briefly explain what the command does and why it works
- If there are important options or variations shown in the docs, mention them
- If the provided documentation does not contain enough information to \
answer the question, say so explicitly rather than guessing or drawing \
on outside knowledge

End your answer with a SOURCES section listing only the chunk_ids you \
drew from, in this exact format:

SOURCES:
- chunk_id: <id> | <title>

Documentation:
{context}"""


def parse_citations(raw_answer, retrieved_chunks):
    valid_ids = {c["chunk_id"] for c in retrieved_chunks}
    cited = []
    if "SOURCES:" in raw_answer:
        sources_section = raw_answer.split("SOURCES:")[1]
        for line in sources_section.strip().split("\n"):
            if "chunk_id:" in line:
                cited_id = line.split("chunk_id:")[1].split("|")[0].strip()
                if cited_id in valid_ids:
                    chunk = corpus[cited_id]
                    cited.append({
                        "chunk_id": cited_id,
                        "title": chunk["title"],
                        "command": chunk["command"],
                        "source_type": chunk["source_type"]
                    })
    return cited

enc = tiktoken.encoding_for_model("gpt-4o-mini")

def count_tokens(text):
    return len(enc.encode(text))

def select_chunks_within_budget(chunks, token_budget=6000):
    selected = []
    used = 0
    for chunk in chunks:
        chunk_tokens = count_tokens(build_context([chunk]))
        if used + chunk_tokens <= token_budget:
            selected.append(chunk)
            used += chunk_tokens
        else:
            continue
    return selected

def ask_gitquest(query, n_results=10, token_budget=6000):
    # Step 1: expand query and retrieve candidate set
    candidates = expand_and_retrieve(query, n_results=n_results)

    # Step 2: rerank candidates
    reranked = rerank(query, candidates, top_n=5)

    # Step 3: apply token budget as a safety net
    final_chunks = select_chunks_within_budget(reranked, token_budget=token_budget)

    # Step 4: build context and generate
    context = build_context(final_chunks)
    response = oai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT.format(context=context)
            },
            {"role": "user", "content": query}
        ]
    )
    raw_answer = response.choices[0].message.content
    citations = parse_citations(raw_answer, final_chunks)
    answer_text = raw_answer.split("SOURCES:")[0].strip()

    return {
        "query": query,
        "answer": answer_text,
        "citations": citations,
        "retrieved_chunks": final_chunks,
        "candidate_count": len(candidates)
    }

In [ ]:
query = "How do I discard all the changes I made to a file and restore it to the last commit?"

def retrieve(query, n_results=5):
    response = co.embed(
        texts=[query],
        model="embed-v4.0",
        input_type="search_query",
        embedding_types=["float"]
    )
    query_embedding = response.embeddings.float[0]

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )

    chunks = []
    for chunk_id, metadata, distance in zip(
        results["ids"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        chunk = corpus[chunk_id]
        chunks.append({
            "chunk_id": chunk_id,
            "text": chunk["text"],
            "title": chunk["title"],
            "source_type": metadata["source_type"],
            "command": metadata["command"],
            "distance": distance
        })
    return chunks

In [ ]:
# testing

chunks = retrieve("How do I discard all the changes I made to a file and restore it to the last commit?")

for chunk in chunks:
    print(f"[{chunk['distance']:.4f}] {chunk['source_type']} | {chunk['title']}")
    print(f"  chunk_id: {chunk['chunk_id']}")
    print(f"  text preview: {chunk['text'][:100]}...")
    print()

In [ ]:
queries = [
    "How do I discard all the changes I made to a file and restore it to the last commit?",
    "What is the difference between git switch and git checkout for changing branches?",
    "What does git stash do and when should I use it?"
]

for query in queries:
    result = ask_gitquest(query)
    print(f"QUERY: {result['query']}\n")
    print(f"ANSWER:\n{result['answer']}\n")
    print("Retrieved chunks:")
    for chunk in result["retrieved_chunks"]:
        print(f"  [{chunk['distance']:.4f}] {chunk['source_type']} | {chunk['title']}")
    print("=" * 60 + "\n")

In [ ]:
result = ask_gitquest("What is the difference between git switch and git checkout for changing branches?")
print(f"QUERY: {result['query']}\n")
print(f"ANSWER:\n{result['answer']}\n")
print("CITATIONS:")
for c in result["citations"]:
    print(f"  [{c['source_type']}] {c['chunk_id']} | {c['title']}")
print(f"\nRetrieved {len(result['retrieved_chunks'])} chunks, cited {len(result['citations'])}")

In [ ]:
queries = [
    "How do I discard all the changes I made to a file and restore it to the last commit?",
    "What is the difference between git switch and git checkout for changing branches?",
    "How do I create a new branch and switch to it at the same time?",
    "What happens to my uncommitted changes when I switch branches?",
    "What does git stash do and when should I use it?",
    "How do I configure Git to use a custom SSH key for a specific host?"
]
"""
The six queries cover a range of scenarios:
    1. single command with a clear modern equivalent

    2. comparison between two related commands

    3. straightforward how-to with a specific syntax answer

    4. question about Git's behavior rather than a specific command

    5. question about a utility command with multiple subcommands

    6. question that falls outside the corpus entirely

review:
- Is the answer grounded in what was actually retrieved?

- Do the citations match the chunks that came back?

- Does GitQuest handle the “unanswerable query”
  (the one about a custom SSH key) correctly?
"""

for query in queries:
    result = ask_gitquest(query)
    print(f"QUERY: {result['query']}\n")
    print(f"ANSWER:\n{result['answer']}\n")
    print("CITATIONS:")
    for c in result["citations"]:
        print(f"  [{c['source_type']}] {c['chunk_id']} | {c['title']}")
    print(f"\nRetrieved {len(result['retrieved_chunks'])} chunks, "
          f"cited {len(result['citations'])}")
    print("=" * 60 + "\n")

In [ ]:
# rag retrieval and context_management
result = ask_gitquest("how do I see the commit history?")
print(result["answer"])
print()
print("Citations:")
for c in result["citations"]:
    print(f"  - {c['chunk_id']} | {c['title']}")
print(f"\nCandidates before reranking: {result['candidate_count']}")

print("\n --------------------------------------------------------------------------------------------------\n")

result = ask_gitquest("how do I see the commit history?")
context_tokens = count_tokens(build_context(result["retrieved_chunks"]))
print(f"Chunks sent to LLM: {len(result['retrieved_chunks'])}")
print(f"Context tokens used: {context_tokens}")
print(f"Token budget remaining: {6000 - context_tokens}")

In [ ]:
# recall_latency.py


import time
from gitquest import ask_gitquest

TEST_QUERIES = [
    ("how do I unstage a file I accidentally added?", "d5237eadab2f87ed"),
    ("how do I undo my last commit but keep my changes?", "ec4d8313b34c85a4"),
    ("how do I go back to a previous version of my file?", "90eb490de11828d8"),
    ("how do I see the commit history?", "eb1e19f70c96b73f"),
    ("how do I create a new branch?", "117d846e6e37c153"),
    ("how do I move commits from one branch to another?", "bc750e0e88a5bb43"),
    ("how do I rename a remote?", "70e8f045b665be41"),
]


def run_experiment(k, n_runs=3):
    total_times = []
    recalls = []

    for query, gold_id in TEST_QUERIES:
        found_in_any_run = False
        query_times = []

        for _ in range(n_runs):
            t0 = time.time()
            result = ask_gitquest(query, n_results=k)
            t1 = time.time()
            query_times.append(t1 - t0)

            if any(c["chunk_id"] == gold_id for c in result["retrieved_chunks"]):
                found_in_any_run = True

        total_times.append(sum(query_times) / n_runs)
        recalls.append(found_in_any_run)

    return {
        "k": k,
        "recall": sum(recalls) / len(recalls),
        "recalls": list(zip([q for q, _ in TEST_QUERIES], recalls)),
        "avg_total": sum(total_times) / len(total_times),
    }


for k in [5, 10, 20]:
    print(f"Running experiment for k={k}...")
    results = run_experiment(k)
    print(f"  Recall@5:  {results['recall']:.0%}")
    print(f"  Avg total: {results['avg_total']:.2f}s")
    print()

